In [32]:
import pandas as pd
import numpy as np

In [33]:
DATA_PATH = "../data/raw/Food_Inspections_20261005.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (316703, 17)


In [34]:
restaurant_df = df[
    df["Facility Type"] == "Restaurant"
].copy()

print("Restaurant records:", len(restaurant_df))
print("Unique licenses:", restaurant_df["License #"].nunique())

Restaurant records: 214845
Unique licenses: 27857


In [35]:
restaurant_history = restaurant_df[
    restaurant_df["License #"].notna()
].copy()

restaurant_history["Inspection Date"] = pd.to_datetime(
    restaurant_history["Inspection Date"]
)

restaurant_history = restaurant_history.sort_values(
    ["License #", "Address", "Inspection Date"]
).reset_index(drop=True)

print("Rows:", len(restaurant_history))
print("Restaurant-location pairs:",
      restaurant_history.groupby(["License #", "Address"]).ngroups)

Rows: 214844
Restaurant-location pairs: 39604


In [36]:
restaurant_history["next_result"] = (
    restaurant_history
    .groupby(["License #", "Address"])["Results"]
    .shift(-1)
)

print("Next-result distribution:")
print(
    restaurant_history["next_result"]
    .value_counts(dropna=False)
)

Next-result distribution:
next_result
Pass                    91722
NaN                     39604
Pass w/ Conditions      29461
Fail                    28597
Out of Business         14031
No Entry                11136
Not Ready                 270
Business Not Located       23
Name: count, dtype: int64


In [37]:
target_mapping = {
    "Fail": 1,
    "Pass": 0,
    "Pass w/ Conditions": 0
}

restaurant_history["target"] = (
    restaurant_history["next_result"].map(target_mapping)
)

print("Target distribution including NaN:")
print(
    restaurant_history["target"]
    .value_counts(dropna=False)
)

Target distribution including NaN:
target
0.0    121183
NaN     65064
1.0     28597
Name: count, dtype: int64


In [38]:
modeling_df = restaurant_history[
    restaurant_history["target"].notna()
].copy()

print("Modeling dataset shape:", modeling_df.shape)
print("\nTarget distribution:")
print(modeling_df["target"].value_counts())

Modeling dataset shape: (149780, 19)

Target distribution:
target
0.0    121183
1.0     28597
Name: count, dtype: int64


In [39]:
duplicate_ids = modeling_df["Inspection ID"].duplicated().sum()

print("Duplicate Inspection IDs:", duplicate_ids)
print("Total rows:", len(modeling_df))
print("Unique Inspection IDs:", modeling_df["Inspection ID"].nunique())

Duplicate Inspection IDs: 0
Total rows: 149780
Unique Inspection IDs: 149780


In [40]:
null_summary = pd.DataFrame({
    "Missing Values": modeling_df.isna().sum(),
    "Missing %": modeling_df.isna().mean() * 100
})

null_summary = null_summary[
    null_summary["Missing Values"] > 0
].sort_values(
    "Missing Values",
    ascending=False
)

print("Columns with missing values:")
print(null_summary)

Columns with missing values:
            Missing Values  Missing %
Violations           27745  18.523835
AKA Name               396   0.264388
Location               144   0.096141
Latitude               144   0.096141
Longitude              144   0.096141
City                    53   0.035385
State                   35   0.023368
Zip                      3   0.002003
Risk                     1   0.000668


In [41]:
license_zero = modeling_df["License #"] == 0

print("Rows with License # = 0:", license_zero.sum())

print("\nTarget distribution:")
print(
    modeling_df.loc[license_zero, "target"]
    .value_counts(dropna=False)
)

Rows with License # = 0: 59

Target distribution:
target
0.0    45
1.0    14
Name: count, dtype: int64


In [42]:
modeling_df = modeling_df[
    modeling_df["License #"] != 0
].copy()

print("Modeling dataset shape:", modeling_df.shape)
print("Rows with License # = 0:", (modeling_df["License #"] == 0).sum())

Modeling dataset shape: (149721, 19)
Rows with License # = 0: 0


In [43]:
print("License dtype:", modeling_df["License #"].dtype)

print("\nNegative License # values:",
      (modeling_df["License #"] < 0).sum())

print("\nUnique License # values:",
      modeling_df["License #"].nunique())

print("\nLicense # summary:")
print(modeling_df["License #"].describe())

License dtype: float64

Negative License # values: 0

Unique License # values: 21124

License # summary:
count    1.497210e+05
mean     1.785860e+06
std      9.097150e+05
min      2.000000e+00
25%      1.447811e+06
50%      2.064468e+06
75%      2.431752e+06
max      3.624677e+06
Name: License #, dtype: float64


In [44]:
print("Zip dtype:", modeling_df["Zip"].dtype)

print("\nNegative Zip values:",
      (modeling_df["Zip"] < 0).sum())

print("\nZip summary:")
print(modeling_df["Zip"].describe())

print("\nSmallest Zip values:")
print(
    modeling_df["Zip"]
    .dropna()
    .sort_values()
    .head(10)
)

print("\nLargest Zip values:")
print(
    modeling_df["Zip"]
    .dropna()
    .sort_values()
    .tail(10)
)

Zip dtype: float64

Negative Zip values: 0

Zip summary:
count    149718.000000
mean      60628.787821
std          19.334588
min       60601.000000
25%       60613.000000
50%       60625.000000
75%       60643.000000
max       60827.000000
Name: Zip, dtype: float64

Smallest Zip values:
253       60601.0
136476    60601.0
136475    60601.0
290       60601.0
289       60601.0
288       60601.0
246       60601.0
245       60601.0
244       60601.0
243       60601.0
Name: Zip, dtype: float64

Largest Zip values:
40093     60827.0
40094     60827.0
40081     60827.0
40082     60827.0
40083     60827.0
40086     60827.0
40091     60827.0
40087     60827.0
40088     60827.0
180442    60827.0
Name: Zip, dtype: float64


In [45]:
print("Latitude summary:")
print(modeling_df["Latitude"].describe())

print("\nLongitude summary:")
print(modeling_df["Longitude"].describe())

print("\nInvalid latitude values:")
print(
    ((modeling_df["Latitude"] < -90) |
     (modeling_df["Latitude"] > 90)).sum()
)

print("\nInvalid longitude values:")
print(
    ((modeling_df["Longitude"] < -180) |
     (modeling_df["Longitude"] > 180)).sum()
)

Latitude summary:
count    149577.000000
mean         41.887176
std           0.076219
min          41.644670
25%          41.852504
50%          41.893349
75%          41.940090
max          42.020808
Name: Latitude, dtype: float64

Longitude summary:
count    149577.000000
mean        -87.675023
std           0.060088
min         -87.906874
25%         -87.703486
50%         -87.662346
75%         -87.633378
max         -87.525094
Name: Longitude, dtype: float64

Invalid latitude values:
0

Invalid longitude values:
0


In [46]:
coord_check = pd.DataFrame({
    "Latitude Missing": modeling_df["Latitude"].isna(),
    "Longitude Missing": modeling_df["Longitude"].isna()
})

print(pd.crosstab(
    coord_check["Latitude Missing"],
    coord_check["Longitude Missing"]
))

Longitude Missing   False  True 
Latitude Missing                
False              149577      0
True                    0    144


In [47]:
expected_target = modeling_df["next_result"].map({
    "Fail": 1,
    "Pass": 0,
    "Pass w/ Conditions": 0
})

mismatch = (
    modeling_df["target"] != expected_target
).sum()

print("Target mismatches:", mismatch)

print("\nNext-result distribution:")
print(modeling_df["next_result"].value_counts())

print("\nTarget distribution:")
print(modeling_df["target"].value_counts())

Target mismatches: 0

Next-result distribution:
next_result
Pass                  91692
Pass w/ Conditions    29446
Fail                  28583
Name: count, dtype: int64

Target distribution:
target
0.0    121138
1.0     28583
Name: count, dtype: int64


In [48]:
print("Missing Inspection Dates:",
      modeling_df["Inspection Date"].isna().sum())

print("\nMinimum Inspection Date:",
      modeling_df["Inspection Date"].min())

print("Maximum Inspection Date:",
      modeling_df["Inspection Date"].max())

print("\nFuture dates:")
print(
    (modeling_df["Inspection Date"] > pd.Timestamp.today()).sum()
)

Missing Inspection Dates: 0

Minimum Inspection Date: 2010-01-04 00:00:00
Maximum Inspection Date: 2026-09-30 00:00:00

Future dates:
0


In [49]:
same_day_counts = (
    modeling_df
    .groupby(["License #", "Address", "Inspection Date"])
    .size()
)

print("Restaurant-location-date groups with multiple inspections:",
      (same_day_counts > 1).sum())

print("\nMaximum inspections on the same date:",
      same_day_counts.max())

print("\nDistribution of same-day inspection counts:")
print(
    same_day_counts[same_day_counts > 1]
    .value_counts()
    .sort_index()
)

Restaurant-location-date groups with multiple inspections: 436

Maximum inspections on the same date: 57

Distribution of same-day inspection counts:
2     410
3       6
4       4
5       4
6       2
7       1
8       1
11      1
15      1
16      1
19      1
25      1
35      1
40      1
57      1
Name: count, dtype: int64


In [50]:
same_day_groups = (
    restaurant_history
    .groupby(["License #", "Address", "Inspection Date"])
    .size()
    .reset_index(name="inspection_count")
)

same_day_groups = same_day_groups[
    same_day_groups["inspection_count"] > 1
].sort_values(
    "inspection_count",
    ascending=False
)

print("Top same-day inspection groups:")
print(same_day_groups.head(10))

Top same-day inspection groups:
       License #                   Address Inspection Date  inspection_count
44843  1354323.0  1410 S MUSEUM CAMPUS DR       2010-06-07                57
44853  1354323.0  1410 S MUSEUM CAMPUS DR       2013-11-14                47
83711  1974745.0        1901 W MADISON ST       2011-02-25                41
44846  1354323.0  1410 S MUSEUM CAMPUS DR       2011-05-24                35
44845  1354323.0  1410 S MUSEUM CAMPUS DR       2011-05-23                25
7754     14616.0            333 W 35TH ST       2011-03-28                19
53700  1574001.0        1060 W ADDISON ST       2011-03-25                16
7750     14616.0            333 W 35TH ST       2010-03-25                15
53697  1574001.0        1060 W ADDISON ST       2010-04-06                11
53696  1574001.0        1060 W ADDISON ST       2010-04-05                 8


In [51]:
example_same_day = restaurant_history[
    (restaurant_history["License #"] == 1354323.0) &
    (restaurant_history["Address"] == "1410 S MUSEUM CAMPUS DR") &
    (restaurant_history["Inspection Date"] == pd.Timestamp("2010-06-07"))
].copy()

print(
    example_same_day[
        [
            "Inspection ID",
            "DBA Name",
            "AKA Name",
            "Inspection Type",
            "Results",
            "Risk",
            "Violations"
        ]
    ].to_string(index=False)
)

Empty DataFrame
Columns: [Inspection ID, DBA Name, AKA Name, Inspection Type, Results, Risk, Violations]
Index: []


In [52]:
# Take the largest same-day inspection group
top_group = same_day_groups.iloc[0]

license_value = top_group["License #"]
address_value = top_group["Address"]
date_value = top_group["Inspection Date"]

print("License:", license_value)
print("Address:", address_value)
print("Date:", date_value)
print("Expected records:", top_group["inspection_count"])

License: 1354323.0
Address: 1410 S MUSEUM CAMPUS DR 
Date: 2010-06-07 00:00:00
Expected records: 57


In [53]:
example_same_day = restaurant_history[
    (restaurant_history["License #"] == license_value) &
    (restaurant_history["Address"].str.strip() == address_value.strip()) &
    (restaurant_history["Inspection Date"].dt.date == date_value.date())
].copy()

print("Records found:", len(example_same_day))

print(
    example_same_day[
        [
            "Inspection ID",
            "DBA Name",
            "AKA Name",
            "Inspection Type",
            "Results",
            "Risk",
            "Violations"
        ]
    ].to_string(index=False)
)

Records found: 57
 Inspection ID                   DBA Name                AKA Name Inspection Type Results            Risk                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [54]:
same_day_analysis = (
    restaurant_history
    .groupby(["License #", "Address", "Inspection Date"])
    .agg(
        inspection_count=("Inspection ID", "count"),
        unique_dba=("DBA Name", "nunique"),
        unique_aka=("AKA Name", "nunique"),
        unique_inspection_type=("Inspection Type", "nunique"),
        unique_results=("Results", "nunique")
    )
    .reset_index()
)

same_day_analysis = same_day_analysis[
    same_day_analysis["inspection_count"] > 1
].copy()

print("Same-day groups:", len(same_day_analysis))

print("\nGroups with multiple DBA names:",
      (same_day_analysis["unique_dba"] > 1).sum())

print("Groups with multiple AKA names:",
      (same_day_analysis["unique_aka"] > 1).sum())

print("Groups with multiple inspection types:",
      (same_day_analysis["unique_inspection_type"] > 1).sum())

print("Groups with multiple results:",
      (same_day_analysis["unique_results"] > 1).sum())

print("\nSummary:")
print(
    same_day_analysis[
        [
            "inspection_count",
            "unique_dba",
            "unique_aka",
            "unique_inspection_type",
            "unique_results"
        ]
    ].describe()
)

Same-day groups: 1137

Groups with multiple DBA names: 390
Groups with multiple AKA names: 414
Groups with multiple inspection types: 534
Groups with multiple results: 867

Summary:
       inspection_count   unique_dba   unique_aka  unique_inspection_type  \
count       1137.000000  1137.000000  1137.000000             1137.000000   
mean           2.276165     1.349164     1.595427                1.473175   
std            2.823020     0.496805     2.427450                0.506500   
min            2.000000     1.000000     0.000000                1.000000   
25%            2.000000     1.000000     1.000000                1.000000   
50%            2.000000     1.000000     1.000000                1.000000   
75%            2.000000     2.000000     2.000000                2.000000   
max           57.000000     5.000000    42.000000                3.000000   

       unique_results  
count     1137.000000  
mean         1.767810  
std          0.434739  
min          1.000000  
25% 

In [55]:
# Create a date-level sequence for each License + Address
date_sequence = (
    restaurant_history
    .groupby(["License #", "Address", "Inspection Date"])
    .agg(
        inspection_count=("Inspection ID", "count"),
        result_count=("Results", "nunique")
    )
    .reset_index()
)

print("Total restaurant-location-date observations:",
      len(date_sequence))

print("Total restaurant-location pairs:",
      date_sequence.groupby(
          ["License #", "Address"]
      ).ngroups)

# Number of restaurant-location-date observations
# that have another distinct date after them
date_sequence = date_sequence.sort_values(
    ["License #", "Address", "Inspection Date"]
)

date_sequence["next_date"] = (
    date_sequence
    .groupby(["License #", "Address"])["Inspection Date"]
    .shift(-1)
)

print("\nDate-level observations with a later inspection date:",
      date_sequence["next_date"].notna().sum())

print("Date-level observations without a later inspection date:",
      date_sequence["next_date"].isna().sum())

Total restaurant-location-date observations: 213393
Total restaurant-location pairs: 39604

Date-level observations with a later inspection date: 173789
Date-level observations without a later inspection date: 39604


In [56]:
# Work at the restaurant-location-date level
date_results = (
    restaurant_history
    .groupby(
        ["License #", "Address", "Inspection Date"]
    )["Results"]
    .agg(
        unique_results="nunique",
        results=lambda x: list(x.dropna().unique())
    )
    .reset_index()
)

# Only dates with multiple different results
mixed_result_dates = date_results[
    date_results["unique_results"] > 1
].copy()

print("Total restaurant-location-date observations:",
      len(date_results))

print("Dates with multiple different Results:",
      len(mixed_result_dates))

print("\nPercentage:",
      round(
          len(mixed_result_dates) / len(date_results) * 100,
          2
      ),
      "%")

print("\nExamples:")
print(
    mixed_result_dates.head(20).to_string(index=False)
)

Total restaurant-location-date observations: 213393
Dates with multiple different Results: 867

Percentage: 0.41 %

Examples:
 License #                   Address Inspection Date  unique_results                               results
    1491.0           755 S CLARK ST       2017-12-27               2 [Out of Business, Pass w/ Conditions]
    1804.0            1160 W 18TH ST      2017-07-17               2            [Pass w/ Conditions, Pass]
    1806.0         310 S HALSTED ST       2017-08-01               2               [Out of Business, Pass]
    1922.0          151 E WACKER DR       2013-07-18               2               [Out of Business, Pass]
    3903.0           3140 W 47TH ST       2014-06-23               2               [Pass, Out of Business]
    4190.0         11 E ILLINOIS ST       2024-11-08               2               [Pass, Out of Business]
    6775.0       3610 N WESTERN AVE       2015-09-30               2               [Pass, Out of Business]
    6856.0        

In [57]:
# Create date-level result information
date_results = (
    restaurant_history
    .groupby(
        ["License #", "Address", "Inspection Date"]
    )["Results"]
    .agg(
        unique_results="nunique",
        results=lambda x: list(x.dropna().unique())
    )
    .reset_index()
)

# Identify dates where multiple different outcomes occurred
ambiguous_dates = date_results[
    date_results["unique_results"] > 1
].copy()

# Add the next distinct inspection date
date_results = date_results.sort_values(
    ["License #", "Address", "Inspection Date"]
)

date_results["next_date"] = (
    date_results
    .groupby(["License #", "Address"])["Inspection Date"]
    .shift(-1)
)

# Find observations whose NEXT date is ambiguous
ambiguous_date_keys = ambiguous_dates[
    ["License #", "Address", "Inspection Date"]
].copy()

ambiguous_date_keys["ambiguous_next_date"] = True

date_results = date_results.merge(
    ambiguous_date_keys,
    left_on=["License #", "Address", "next_date"],
    right_on=["License #", "Address", "Inspection Date"],
    how="left"
)

date_results["ambiguous_next_date"] = (
    date_results["ambiguous_next_date"]
    .fillna(False)
)

print("Prediction dates with an ambiguous next date:",
      date_results["ambiguous_next_date"].sum())

print(
    "Percentage of prediction dates:",
    round(
        date_results["ambiguous_next_date"].mean() * 100,
        2
    ),
    "%"
)

Prediction dates with an ambiguous next date: 788
Percentage of prediction dates: 0.37 %


In [58]:
# Count prediction dates before removing ambiguous next dates
total_prediction_dates = date_results["next_date"].notna().sum()

# Remove prediction dates whose next inspection date is ambiguous
clean_date_results = date_results[
    date_results["next_date"].notna() &
    (~date_results["ambiguous_next_date"])
].copy()

print("Total prediction dates:", total_prediction_dates)
print("Ambiguous next dates excluded:",
      date_results["ambiguous_next_date"].sum())
print("Clean prediction dates:",
      len(clean_date_results))

Total prediction dates: 173789
Ambiguous next dates excluded: 788
Clean prediction dates: 173789


In [59]:
print("Ambiguous flag distribution:")
print(date_results["ambiguous_next_date"].value_counts(dropna=False))

print("\nAmbiguous next dates:")
print(date_results["ambiguous_next_date"].sum())

print("\nRows with next date:")
print(date_results["next_date"].notna().sum())

Ambiguous flag distribution:
ambiguous_next_date
False    212605
True        788
Name: count, dtype: int64

Ambiguous next dates:
788

Rows with next date:
173789


In [60]:
clean_date_results = date_results[
    date_results["next_date"].notna()
].copy()

clean_date_results = clean_date_results[
    clean_date_results["ambiguous_next_date"] == False
].copy()

print("Total prediction dates:", total_prediction_dates)
print("Ambiguous next dates excluded:",
      date_results["ambiguous_next_date"].sum())
print("Clean prediction dates:", len(clean_date_results))

Total prediction dates: 173789
Ambiguous next dates excluded: 788
Clean prediction dates: 173001


In [61]:
print(date_results.columns.tolist())

['License #', 'Address', 'Inspection Date_x', 'unique_results', 'results', 'next_date', 'Inspection Date_y', 'ambiguous_next_date']


In [62]:
# Create a clean lookup table:
# one row per restaurant-location-date
date_result_lookup = (
    restaurant_history
    .groupby(
        ["License #", "Address", "Inspection Date"]
    )["Results"]
    .agg(
        unique_results="nunique",
        results=lambda x: list(x.dropna().unique())
    )
    .reset_index()
)

# Keep only clean prediction dates
clean_date_results = date_results[
    date_results["next_date"].notna() &
    (date_results["ambiguous_next_date"] == False)
].copy()

# Attach the result from the next inspection date
clean_date_results = clean_date_results.merge(
    date_result_lookup[
        [
            "License #",
            "Address",
            "Inspection Date",
            "results"
        ]
    ],
    left_on=[
        "License #",
        "Address",
        "next_date"
    ],
    right_on=[
        "License #",
        "Address",
        "Inspection Date"
    ],
    how="left"
)

# Rename the matched result
clean_date_results = clean_date_results.rename(
    columns={"results": "next_result"}
)

print("Clean prediction dates:", len(clean_date_results))

print("\nNext inspection result distribution:")
print(
    clean_date_results["next_result"]
    .explode()
    .value_counts(dropna=False)
)

Clean prediction dates: 173001

Next inspection result distribution:


KeyError: 'next_result'

In [ ]:
print("Next result column type:")
print(clean_date_results["next_result"].dtype)

print("\nFirst 10 next_result values:")
print(clean_date_results["next_result"].head(10).to_list())

print("\nMissing next_result:")
print(clean_date_results["next_result"].isna().sum())

Next result column type:


KeyError: 'next_result'

In [ ]:
print("Current columns:")
print(clean_date_results.columns.tolist())

Current columns:
['License #', 'Address', 'Inspection Date_x', 'unique_results', 'results_x', 'next_date', 'Inspection Date_y', 'ambiguous_next_date', 'Inspection Date', 'results_y']


In [ ]:
# The result from the next inspection date is stored in results_y
clean_date_results["next_result"] = clean_date_results["results_y"]

# Since non-ambiguous dates should have exactly one result,
# convert the single-item lists into plain strings.
clean_date_results["next_result"] = (
    clean_date_results["next_result"]
    .apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)
)

print("Clean prediction dates:", len(clean_date_results))

print("\nNext inspection result distribution:")
print(
    clean_date_results["next_result"]
    .value_counts(dropna=False)
)

Clean prediction dates: 173001

Next inspection result distribution:
next_result
Pass                    90809
Pass w/ Conditions      29197
Fail                    28369
Out of Business         13254
No Entry                11085
Not Ready                 265
Business Not Located       22
Name: count, dtype: int64


In [ ]:
# Define the binary target
target_mapping = {
    "Fail": 1,
    "Pass": 0,
    "Pass w/ Conditions": 0
}

clean_date_results["target"] = (
    clean_date_results["next_result"]
    .map(target_mapping)
)

# Keep only valid binary targets
modeling_dates = clean_date_results[
    clean_date_results["target"].notna()
].copy()

print("Final modeling prediction dates:", len(modeling_dates))

print("\nTarget distribution:")
print(
    modeling_dates["target"]
    .value_counts()
    .sort_index()
)

print("\nTarget percentage:")
print(
    modeling_dates["target"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

Final modeling prediction dates: 148375

Target distribution:
target
0.0    120006
1.0     28369
Name: count, dtype: int64

Target percentage:
target
0.0    80.880202
1.0    19.119798
Name: proportion, dtype: float64


In [ ]:
print("Target vs Next Result:")
print(
    pd.crosstab(
        modeling_dates["next_result"],
        modeling_dates["target"]
    )
)

Target vs Next Result:
target                0.0    1.0
next_result                     
Fail                    0  28369
Pass                90809      0
Pass w/ Conditions  29197      0


In [ ]:
prediction_keys = [
    "License #",
    "Address",
    "Inspection Date"
]

duplicate_prediction_dates = modeling_dates.duplicated(
    subset=prediction_keys
).sum()

print("Duplicate prediction dates:", duplicate_prediction_dates)
print("Total modeling instances:", len(modeling_dates))

Duplicate prediction dates: 0
Total modeling instances: 148375


In [ ]:
print(modeling_dates.columns.tolist())

['License #', 'Address', 'Inspection Date_x', 'unique_results', 'results_x', 'next_date', 'Inspection Date_y', 'ambiguous_next_date', 'Inspection Date', 'results_y', 'next_result', 'target']


In [ ]:
# Columns from the original inspection record
original_columns = [
    "Inspection ID",
    "DBA Name",
    "AKA Name",
    "License #",
    "Facility Type",
    "Risk",
    "Address",
    "City",
    "State",
    "Zip",
    "Inspection Date",
    "Inspection Type",
    "Results",
    "Violations",
    "Latitude",
    "Longitude",
    "Location"
]

# Keep only the valid prediction dates and target information
target_info = modeling_dates[
    [
        "License #",
        "Address",
        "Inspection Date",
        "next_date",
        "next_result",
        "target"
    ]
].copy()

# Connect target information back to the original inspection record
modeling_handoff = restaurant_history[
    original_columns
].merge(
    target_info,
    on=[
        "License #",
        "Address",
        "Inspection Date"
    ],
    how="inner"
)

print("Handoff dataset shape:", modeling_handoff.shape)

print("\nColumns:")
print(modeling_handoff.columns.tolist())

print("\nTarget distribution:")
print(modeling_handoff["target"].value_counts())

Handoff dataset shape: (148642, 20)

Columns:
['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'City', 'State', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude', 'Location', 'next_date', 'next_result', 'target']

Target distribution:
target
0.0    120250
1.0     28392
Name: count, dtype: int64


In [ ]:
# Check how many prediction dates were duplicated
handoff_keys = [
    "License #",
    "Address",
    "Inspection Date"
]

duplicate_handoff = (
    modeling_handoff
    .groupby(handoff_keys)
    .size()
    .reset_index(name="row_count")
)

duplicate_handoff = duplicate_handoff[
    duplicate_handoff["row_count"] > 1
]

print("Prediction dates with multiple original rows:",
      len(duplicate_handoff))

print("\nExtra rows caused by same-date records:",
      duplicate_handoff["row_count"].sum() - len(duplicate_handoff))

print("\nMaximum original rows for one prediction date:",
      duplicate_handoff["row_count"].max())

Prediction dates with multiple original rows: 169

Extra rows caused by same-date records: 267

Maximum original rows for one prediction date: 35


In [ ]:
target_table = modeling_dates[
    [
        "License #",
        "Address",
        "Inspection Date",
        "next_date",
        "next_result",
        "target"
    ]
].copy()

print("Target table shape:", target_table.shape)

print("\nDuplicate prediction dates:")
print(
    target_table.duplicated(
        subset=["License #", "Address", "Inspection Date"]
    ).sum()
)

print("\nTarget distribution:")
print(target_table["target"].value_counts())

Target table shape: (148375, 6)

Duplicate prediction dates:
0

Target distribution:
target
0.0    120006
1.0     28369
Name: count, dtype: int64


In [ ]:
TARGET_PATH = "../data/processed/restaurant_inspection_targets.csv"

target_table.to_csv(
    TARGET_PATH,
    index=False
)

print("Target table saved successfully!")
print("Path:", TARGET_PATH)
print("Rows:", len(target_table))
print("Columns:", target_table.shape[1])

Target table saved successfully!
Path: ../data/processed/restaurant_inspection_targets.csv
Rows: 148375
Columns: 6


In [ ]:
# Keep the original restaurant inspection records
history_columns = [
    "Inspection ID",
    "DBA Name",
    "AKA Name",
    "License #",
    "Facility Type",
    "Risk",
    "Address",
    "City",
    "State",
    "Zip",
    "Inspection Date",
    "Inspection Type",
    "Results",
    "Violations",
    "Latitude",
    "Longitude",
    "Location"
]

restaurant_history_clean = restaurant_history[
    history_columns
].copy()

# Confirm invalid license IDs are removed
print("History shape:", restaurant_history_clean.shape)
print(
    "License # = 0:",
    (restaurant_history_clean["License #"] == 0).sum()
)
print(
    "Missing License #:",
    restaurant_history_clean["License #"].isna().sum()
)

History shape: (214844, 17)
License # = 0: 208
Missing License #: 0


In [ ]:
# Remove invalid License # values
restaurant_history_clean = restaurant_history_clean[
    restaurant_history_clean["License #"] != 0
].copy()

print("Clean history shape:", restaurant_history_clean.shape)

print(
    "License # = 0:",
    (restaurant_history_clean["License #"] == 0).sum()
)

print(
    "Missing License #:",
    restaurant_history_clean["License #"].isna().sum()
)

Clean history shape: (214636, 17)
License # = 0: 0
Missing License #: 0


In [ ]:
# Check whether every target prediction date exists
# in the cleaned inspection history

history_keys = restaurant_history_clean[
    ["License #", "Address", "Inspection Date"]
].drop_duplicates()

target_keys = target_table[
    ["License #", "Address", "Inspection Date"]
].drop_duplicates()

check = target_keys.merge(
    history_keys,
    on=["License #", "Address", "Inspection Date"],
    how="left",
    indicator=True
)

print("Target prediction dates:", len(target_keys))

print(
    "Prediction dates found in history:",
    (check["_merge"] == "both").sum()
)

print(
    "Prediction dates missing from history:",
    (check["_merge"] == "left_only").sum()
)

Target prediction dates: 148375
Prediction dates found in history: 148316
Prediction dates missing from history: 59


In [ ]:
# Remove invalid license IDs from the target table
target_table = target_table[
    target_table["License #"] != 0
].copy()

print("Clean target table shape:", target_table.shape)

print(
    "License # = 0:",
    (target_table["License #"] == 0).sum()
)

print("\nTarget distribution:")
print(
    target_table["target"].value_counts()
)

Clean target table shape: (148316, 6)
License # = 0: 0

Target distribution:
target
0.0    119961
1.0     28355
Name: count, dtype: int64


In [ ]:
history_keys = restaurant_history_clean[
    ["License #", "Address", "Inspection Date"]
].drop_duplicates()

target_keys = target_table[
    ["License #", "Address", "Inspection Date"]
].drop_duplicates()

check = target_keys.merge(
    history_keys,
    on=["License #", "Address", "Inspection Date"],
    how="left",
    indicator=True
)

print("Target prediction dates:", len(target_keys))

print(
    "Prediction dates found in history:",
    (check["_merge"] == "both").sum()
)

print(
    "Prediction dates missing from history:",
    (check["_merge"] == "left_only").sum()
)

Target prediction dates: 148316
Prediction dates found in history: 148316
Prediction dates missing from history: 0


In [ ]:
TARGET_PATH = "../data/processed/restaurant_inspection_targets.csv"
HISTORY_PATH = "../data/processed/restaurant_inspection_history.csv"

# Save corrected target table
target_table.to_csv(
    TARGET_PATH,
    index=False
)

# Save clean inspection history
restaurant_history_clean.to_csv(
    HISTORY_PATH,
    index=False
)

print("Files saved successfully!")

print("\nTarget table:")
print("Path:", TARGET_PATH)
print("Shape:", target_table.shape)

print("\nInspection history:")
print("Path:", HISTORY_PATH)
print("Shape:", restaurant_history_clean.shape)

Files saved successfully!

Target table:
Path: ../data/processed/restaurant_inspection_targets.csv
Shape: (148316, 6)

Inspection history:
Path: ../data/processed/restaurant_inspection_history.csv
Shape: (214636, 17)


In [ ]:
same_date_check = (
    target_table["Inspection Date"] == target_table["next_date"]
).sum()

print("Rows where Inspection Date == next_date:", same_date_check)

Rows where Inspection Date == next_date: 148316


In [ ]:
print(target_table.columns.tolist())

print("\nFirst 10 rows:")
print(
    target_table[
        ["License #", "Address", "Inspection Date", "next_date", "next_result", "target"]
    ].head(10)
)

['License #', 'Address', 'Inspection Date', 'next_date', 'next_result', 'target']

First 10 rows:
    License #           Address Inspection Date  next_date  \
90        2.0  230 W MONROE ST       2011-02-15 2011-02-15   
91        2.0  230 W MONROE ST       2012-03-21 2012-03-21   
92        2.0  230 W MONROE ST       2013-07-19 2013-07-19   
93        2.0  230 W MONROE ST       2014-04-18 2014-04-18   
94        2.0  230 W MONROE ST       2014-11-20 2014-11-20   
95        2.0  230 W MONROE ST       2015-04-21 2015-04-21   
96        2.0  230 W MONROE ST       2015-12-14 2015-12-14   
97        2.0  230 W MONROE ST       2016-03-22 2016-03-22   
98        2.0  230 W MONROE ST       2016-10-27 2016-10-27   
99        2.0  230 W MONROE ST       2016-11-04 2016-11-04   

           next_result  target  
90                Pass     0.0  
91                Pass     0.0  
92                Pass     0.0  
93                Pass     0.0  
94                Pass     0.0  
95                Pas

In [ ]:
print(clean_date_results.columns.tolist())

print("\nFirst 10 rows:")
print(
    clean_date_results[
        ["License #", "Address", "Inspection Date_x", "next_date", "next_result"]
    ].head(10)
)

['License #', 'Address', 'Inspection Date_x', 'unique_results', 'results_x', 'next_date', 'Inspection Date_y', 'ambiguous_next_date', 'Inspection Date', 'results_y', 'next_result', 'target']

First 10 rows:
   License #                Address Inspection Date_x  next_date  \
0        0.0     10148 S EWING AVE         2020-11-20 2021-07-06   
1        0.0  10606 S TORRENCE AVE         2012-06-07 2013-09-09   
2        0.0         109 E 51ST ST         2017-06-02 2017-06-09   
3        0.0        1100 E 55TH ST         2022-03-23 2022-04-01   
4        0.0      11601 W TOUHY AVE        2016-02-11 2025-08-25   
5        0.0     11601 W TOUHY AVE         2015-06-09 2015-06-23   
6        0.0     11601 W TOUHY AVE         2015-06-23 2015-07-07   
7        0.0     11601 W TOUHY AVE         2015-07-07 2015-07-08   
8        0.0     11601 W TOUHY AVE         2015-07-08 2015-08-24   
9        0.0     11601 W TOUHY AVE         2015-08-24 2024-06-05   

          next_result  
0     Out of Busines

In [ ]:
target_table = clean_date_results[
    [
        "License #",
        "Address",
        "Inspection Date_x",
        "next_date",
        "next_result",
        "target"
    ]
].copy()

# Rename the prediction-date column correctly
target_table = target_table.rename(
    columns={"Inspection Date_x": "Inspection Date"}
)

# Remove invalid license numbers
target_table = target_table[
    target_table["License #"] != 0
].copy()

# Reset index
target_table = target_table.reset_index(drop=True)

print("Shape:", target_table.shape)
print("\nColumns:")
print(target_table.columns.tolist())

print("\nFirst 10 rows:")
print(target_table.head(10))

Shape: (172911, 6)

Columns:
['License #', 'Address', 'Inspection Date', 'next_date', 'next_result', 'target']

First 10 rows:
   License #           Address Inspection Date  next_date         next_result  \
0        2.0  230 W MONROE ST       2010-06-15 2011-02-15                Pass   
1        2.0  230 W MONROE ST       2011-02-15 2012-03-21                Pass   
2        2.0  230 W MONROE ST       2012-03-21 2013-07-19                Pass   
3        2.0  230 W MONROE ST       2013-07-19 2014-04-18                Pass   
4        2.0  230 W MONROE ST       2014-04-18 2014-11-20                Pass   
5        2.0  230 W MONROE ST       2014-11-20 2015-04-21                Pass   
6        2.0  230 W MONROE ST       2015-04-21 2015-12-14                Pass   
7        2.0  230 W MONROE ST       2015-12-14 2016-03-22  Pass w/ Conditions   
8        2.0  230 W MONROE ST       2016-03-22 2016-10-27                Fail   
9        2.0  230 W MONROE ST       2016-10-27 2016-11-04      

In [ ]:
same_date_check = (
    target_table["Inspection Date"] == target_table["next_date"]
).sum()

invalid_order_check = (
    target_table["Inspection Date"] >= target_table["next_date"]
).sum()

print("Inspection Date == next_date:", same_date_check)
print("Inspection Date >= next_date:", invalid_order_check)

Inspection Date == next_date: 0
Inspection Date >= next_date: 0


In [ ]:
TARGET_PATH = "../data/processed/restaurant_inspection_targets.csv"

target_table.to_csv(
    TARGET_PATH,
    index=False
)

print("Target table saved successfully!")
print("Path:", TARGET_PATH)
print("Shape:", target_table.shape)

print("\nTarget distribution:")
print(target_table["target"].value_counts().sort_index())

Target table saved successfully!
Path: ../data/processed/restaurant_inspection_targets.csv
Shape: (172911, 6)

Target distribution:
target
0.0    119961
1.0     28355
Name: count, dtype: int64


In [ ]:
target_table = target_table[
    target_table["target"].notna()
].copy()

target_table = target_table.reset_index(drop=True)

print("Final target table shape:", target_table.shape)

print("\nTarget distribution:")
print(target_table["target"].value_counts().sort_index())

print("\nMissing target values:")
print(target_table["target"].isna().sum())

Final target table shape: (148316, 6)

Target distribution:
target
0.0    119961
1.0     28355
Name: count, dtype: int64

Missing target values:
0


In [ ]:
TARGET_PATH = "../data/processed/restaurant_inspection_targets.csv"

target_table.to_csv(
    TARGET_PATH,
    index=False
)

print("Final target table saved successfully!")
print("Path:", TARGET_PATH)
print("Shape:", target_table.shape)

Final target table saved successfully!
Path: ../data/processed/restaurant_inspection_targets.csv
Shape: (148316, 6)


In [ ]:
saved_target = pd.read_csv(TARGET_PATH)

print("Saved file shape:", saved_target.shape)
print("\nColumns:")
print(saved_target.columns.tolist())

print("\nTarget distribution:")
print(saved_target["target"].value_counts().sort_index())

print(
    "\nMissing target values:",
    saved_target["target"].isna().sum()
)

print(
    "Invalid date relationship:",
    (pd.to_datetime(saved_target["Inspection Date"]) >=
     pd.to_datetime(saved_target["next_date"])).sum()
)

Saved file shape: (148316, 6)

Columns:
['License #', 'Address', 'Inspection Date', 'next_date', 'next_result', 'target']

Target distribution:
target
0.0    119961
1.0     28355
Name: count, dtype: int64

Missing target values: 0
Invalid date relationship: 0


In [ ]:
TARGET_PATH = "../data/processed/restaurant_inspection_targets.csv"
HISTORY_PATH = "../data/processed/restaurant_inspection_history.csv"

targets_check = pd.read_csv(TARGET_PATH)
history_check = pd.read_csv(HISTORY_PATH)

targets_check["Inspection Date"] = pd.to_datetime(
    targets_check["Inspection Date"]
)
targets_check["next_date"] = pd.to_datetime(
    targets_check["next_date"]
)
history_check["Inspection Date"] = pd.to_datetime(
    history_check["Inspection Date"]
)

print("Target table shape:", targets_check.shape)
print("History table shape:", history_check.shape)

print("\nTarget columns:")
print(targets_check.columns.tolist())

print("\nHistory columns:")
print(history_check.columns.tolist())

Target table shape: (148316, 6)
History table shape: (214636, 17)

Target columns:
['License #', 'Address', 'Inspection Date', 'next_date', 'next_result', 'target']

History columns:
['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'City', 'State', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude', 'Location']


In [ ]:
# Create the inspection-date keys from the history file
history_keys = history_check[
    ["License #", "Address", "Inspection Date"]
].drop_duplicates()

# Create the prediction-date keys from the target file
target_keys = targets_check[
    ["License #", "Address", "Inspection Date"]
].drop_duplicates()

# Check whether every target prediction date exists in history
date_alignment = target_keys.merge(
    history_keys,
    on=["License #", "Address", "Inspection Date"],
    how="left",
    indicator=True
)

print("Target prediction keys:", len(target_keys))
print(
    "Found in history:",
    (date_alignment["_merge"] == "both").sum()
)
print(
    "Missing from history:",
    (date_alignment["_merge"] == "left_only").sum()
)

Target prediction keys: 148316
Found in history: 148316
Missing from history: 0


In [ ]:
next_date_keys = targets_check[
    ["License #", "Address", "next_date"]
].drop_duplicates()

history_date_keys = history_check[
    ["License #", "Address", "Inspection Date"]
].drop_duplicates()

next_alignment = next_date_keys.merge(
    history_date_keys,
    left_on=["License #", "Address", "next_date"],
    right_on=["License #", "Address", "Inspection Date"],
    how="left",
    indicator=True
)

print("Target next-date keys:", len(next_date_keys))

print(
    "Next dates found in history:",
    (next_alignment["_merge"] == "both").sum()
)

print(
    "Next dates missing from history:",
    (next_alignment["_merge"] == "left_only").sum()
)

Target next-date keys: 148316
Next dates found in history: 148316
Next dates missing from history: 0


In [ ]:
date_order_check = (
    targets_check["Inspection Date"] >= targets_check["next_date"]
)

print("Rows where Inspection Date >= next_date:", date_order_check.sum())

print(
    "Rows where Inspection Date < next_date:",
    (~date_order_check).sum()
)

if date_order_check.sum() == 0:
    print("\n✅ All next inspections occur strictly after the prediction date.")
else:
    print("\n❌ Some rows have an invalid date relationship.")

Rows where Inspection Date >= next_date: 0
Rows where Inspection Date < next_date: 148316

✅ All next inspections occur strictly after the prediction date.


In [ ]:
# Get the actual results recorded at each restaurant-location-date
history_results = (
    history_check
    .groupby(["License #", "Address", "Inspection Date"])["Results"]
    .agg(
        unique_results="nunique",
        results=lambda x: list(x.dropna().unique())
    )
    .reset_index()
)

# Attach the actual next-date result information to the target table
result_check = targets_check.merge(
    history_results,
    left_on=["License #", "Address", "next_date"],
    right_on=["License #", "Address", "Inspection Date"],
    how="left"
)

print("Target rows:", len(result_check))

print(
    "Next dates with exactly one recorded result:",
    (result_check["unique_results"] == 1).sum()
)

print(
    "Next dates with multiple recorded results:",
    (result_check["unique_results"] > 1).sum()
)

print(
    "Next dates with no recorded result:",
    result_check["unique_results"].isna().sum()
)

Target rows: 148316
Next dates with exactly one recorded result: 148316
Next dates with multiple recorded results: 0
Next dates with no recorded result: 0


In [ ]:
# Extract the single actual result for each next inspection date
result_check["actual_next_result"] = result_check["results"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x
)

# Compare target next_result with the actual history result
result_mismatch = (
    result_check["next_result"] != result_check["actual_next_result"]
)

print("Total target rows:", len(result_check))
print("Matching next_result:", (~result_mismatch).sum())
print("Mismatching next_result:", result_mismatch.sum())

Total target rows: 148316
Matching next_result: 148316
Mismatching next_result: 0


In [63]:
location_nulls = pd.DataFrame({
    "Missing Values": restaurant_history[
        ["Location", "Latitude", "Longitude"]
    ].isna().sum(),
    "Missing %": restaurant_history[
        ["Location", "Latitude", "Longitude"]
    ].isna().mean() * 100
})

print(location_nulls)

           Missing Values  Missing %
Location              220     0.1024
Latitude              220     0.1024
Longitude             220     0.1024
